In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(device)

spike_df = pd.read_csv('../../Data_processed/spike_tensors_x.csv')
print(spike_df.head())

cuda
   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3     [2, 2, 3]       [3, 3, 4]         [0.663, 0.493, 0.886]   
1          2     [3, 2, 3]          [4, 5]                [0.933, 1.306]   
2          2     [2, 2, 3]          [6, 9]                [1.191, 1.085]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]       [3, 3, 5]         [0.408, 0.991, 1.413]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 

In [9]:
import pandas as pd
import numpy as np

def find_closest_rows_matrix(df, id_statistics, weather_list, date_list):
    # Convert columns to NumPy arrays
    id_stats_array = np.array(df['ID-statistics'].apply(eval).tolist())
    weather_array = np.array(df['weather'].apply(eval).tolist())
    date_sin_cos_array = df[['date_sin', 'date_cos']].values.astype(np.float64)

    # Stack weather and date_sin_cos for parallel processing
    weather_stack = np.array(weather_list)
    date_stack = np.array(date_list)

    # Calculate the Euclidean distance for each component in parallel
    id_stats_distance = np.linalg.norm(id_stats_array - np.array(id_statistics), axis=1)
    weather_distance_matrix = np.linalg.norm(weather_array[:, None, :] - weather_stack, axis=2)
    date_distance_matrix = np.linalg.norm(date_sin_cos_array[:, None, :] - date_stack, axis=2)

    # Total distance
    total_distance_matrix = id_stats_distance[:, None] + weather_distance_matrix + date_distance_matrix

    # Find the indices of the minimum distance for each day
    closest_indices = np.argmin(total_distance_matrix, axis=0)

    # Extract the closest rows' spike-related information
    spike_nums = df.iloc[closest_indices]['spike_num'].values
    spike_durations = df.iloc[closest_indices]['spike_durations'].apply(eval).tolist()
    spike_types = df.iloc[closest_indices]['spike_type'].apply(eval).tolist()
    spike_magnitudes = df.iloc[closest_indices]['spike_magnitudes'].apply(eval).tolist()
    spike_times_intervals = df.iloc[closest_indices]['spike_times_intervals'].apply(eval).tolist()
    
    return spike_nums, spike_durations, spike_types, spike_magnitudes, spike_times_intervals

def get_spike_df_for_multiple_days(id_statistics, weather_list, date_list):
    df = pd.read_csv('../../Data_processed/spike_tensors_x.csv')
    
    # Get the closest rows for all days in one call
    spike_nums, spike_durations, spike_types, spike_magnitudes, spike_times_intervals = find_closest_rows_matrix(df, id_statistics, weather_list, date_list)
    
    all_days_spike_df = pd.DataFrame()
    
    for day_idx in range(len(weather_list)):
        spike_df = pd.DataFrame()
        spike_df['time'] = pd.date_range('00:00:00', '23:30:00', freq='30T').time
        spike_df['spike_type'] = 0
        spike_df['spike_magnitude'] = 1.0

        for i in range(spike_nums[day_idx]):
            spike_start = spike_times_intervals[day_idx][i]
            spike_duration = spike_durations[day_idx][i]
            spike_end = spike_start + spike_duration

            if spike_start + 1 < spike_end - 1:
                spike_center = np.random.randint(spike_start + 1, spike_end - 1)
            else:
                spike_center = spike_start + (spike_duration // 2)

            spike_df.loc[spike_start:spike_center-1, 'spike_type'] = 1
            spike_df.loc[spike_center, 'spike_type'] = spike_types[day_idx][i]
            spike_df.loc[spike_center, 'spike_magnitude'] = spike_magnitudes[day_idx][i]
            spike_df.loc[spike_center+1:spike_end-1, 'spike_type'] = 4

        all_days_spike_df = pd.concat([all_days_spike_df, spike_df], axis=0, ignore_index=True)

    return all_days_spike_df

# Example usage
id_statistics = [0.25259, 0.158, 0.24707, 0.0, 2.994, 0.109261]
weather_list = [[0.49, 0.374875, 0.759], [0.5, 0.4, 0.8], [0.55, 0.47, 0.82]]
date_list = [[-0.95, 0.29], [-0.93, 0.26], [-0.92, 0.27]]

multiple_days_spike_df = get_spike_df_for_multiple_days(id_statistics, weather_list, date_list)
print(multiple_days_spike_df)


         time  spike_type  spike_magnitude
0    00:00:00           0            1.000
1    00:30:00           0            1.000
2    01:00:00           0            1.000
3    01:30:00           0            1.000
4    02:00:00           0            1.000
..        ...         ...              ...
139  21:30:00           1            1.000
140  22:00:00           2            0.377
141  22:30:00           4            1.000
142  23:00:00           0            1.000
143  23:30:00           0            1.000

[144 rows x 3 columns]


In [18]:
import pandas as pd
import numpy as np

def date_to_sin_cos(date):
    """Convert a date to its sine and cosine representation."""
    day_of_year = date.timetuple().tm_yday
    total_days = 365 + (1 if date.year % 4 == 0 else 0)  # Account for leap years
    angle = 2 * np.pi * (day_of_year / total_days)
    date_sin = np.sin(angle)
    date_cos = np.cos(angle)
    return date_sin, date_cos

def generate_date_sin_cos_list(start_date, end_date):
    """Generate a list of sine-cosine values for all dates between start_date and end_date."""
    # Generate date range
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    
    # Convert each date to its sine-cosine form
    sin_cos_list = [date_to_sin_cos(date) for date in date_range]
    
    return sin_cos_list

# Example usage
start_date = '2013-11-11 00:00:00 00:00:00'
end_date = '2013-11-13 00:00:00 23:30:00'

date_list = generate_date_sin_cos_list(start_date, end_date)
print(date_list)

id_statistics = [0.25, 0.158, 0.24, 0.0, 2.994, 0.1]
weather_list = [[0.49, 0.374875, 0.759], [0.5, 0.4, 0.8], [0.55, 0.47, 0.82]]

multiple_days_spike_df = get_spike_df_for_multiple_days(id_statistics, weather_list, date_list)
print(multiple_days_spike_df)

[(-0.7583058084785625, 0.6518989958787125), (-0.7469720876965558, 0.6648553979642859), (-0.7354170229639858, 0.6776147890466887)]
         time  spike_type  spike_magnitude
0    00:00:00           0              1.0
1    00:30:00           0              1.0
2    01:00:00           0              1.0
3    01:30:00           0              1.0
4    02:00:00           0              1.0
..        ...         ...              ...
139  21:30:00           0              1.0
140  22:00:00           0              1.0
141  22:30:00           0              1.0
142  23:00:00           0              1.0
143  23:30:00           0              1.0

[144 rows x 3 columns]


In [19]:
# print out the multiple_days_spike_df where spike_type is not 0
print(multiple_days_spike_df[multiple_days_spike_df['spike_type'] != 0])

         time  spike_type  spike_magnitude
67   09:30:00           1            1.000
68   10:00:00           1            1.000
69   10:30:00           1            1.000
70   11:00:00           2            1.862
71   11:30:00           4            1.000
72   12:00:00           4            1.000
73   12:30:00           4            1.000
74   13:00:00           4            1.000
75   13:30:00           4            1.000
76   14:00:00           4            1.000
86   19:00:00           1            1.000
87   19:30:00           2            1.052
88   20:00:00           4            1.000
89   20:30:00           4            1.000
117  10:30:00           1            1.000
118  11:00:00           2            0.426
119  11:30:00           4            1.000
121  12:30:00           1            1.000
122  13:00:00           2            0.435
123  13:30:00           4            1.000
125  14:30:00           1            1.000
126  15:00:00           2            0.594
127  15:30:

In [21]:
import pandas as pd

# Assuming weather_df is already loaded with the provided data
def daily_weather_means(weather_df):
    # Convert the 'tstp' column to datetime if it's not already
    weather_df['tstp'] = pd.to_datetime(weather_df['tstp'])
    
    # Extract the date from the timestamp
    weather_df['date'] = weather_df['tstp'].dt.date
    
    # Group by date and calculate the mean for each group
    daily_means = weather_df.groupby('date').agg({
        'temperature': 'mean',
        'windSpeed': 'mean',
        'humidity': 'mean'
    }).reset_index()
    
    # Convert the resulting DataFrame into a list of lists
    weather_list = daily_means[['temperature', 'windSpeed', 'humidity']].values.tolist()
    
    return weather_list

# Example usage
weather_df = pd.DataFrame({
    'tstp': [
        '2014-01-31 00:00:00', '2014-01-31 00:30:00', '2014-01-31 01:00:00', 
        '2014-01-31 01:30:00', '2014-01-31 02:00:00', '2014-02-02 21:00:00', 
        '2014-02-02 21:30:00', '2014-02-02 22:00:00', '2014-02-02 22:30:00', 
        '2014-02-02 23:00:00'
    ],
    'temperature': [0.1561, 0.1232, 0.0903, 0.0884, 0.0865, 0.5510, 0.5303, 0.5097, 0.5013, 0.4929],
    'windSpeed': [0.0656, 0.0822, 0.0989, 0.0977, 0.0965, 0.3278, 0.3147, 0.3015, 0.2896, 0.2777],
    'humidity': [0.9250, 0.9625, 1.0000, 1.0000, 1.0000, 0.7000, 0.7000, 0.7000, 0.7375, 0.7750]
})

weather_list = daily_weather_means(weather_df)
print(weather_list)


[[0.1089, 0.08818000000000001, 0.9775], [0.5170399999999999, 0.30226, 0.7224999999999999]]
